# Preprocessed Data Set 
#### (data cleaning)
---
### Necessary imports

In [41]:
import pandas as pd
import re
import string
from pathlib import Path

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adithi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/adithi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/adithi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [42]:
DATA_PATH = Path("../data/raw_data/WELFake_Dataset.csv")

df = pd.read_csv(DATA_PATH)

### Handle the missing values and any duplicates

In [43]:
# Fill missing titles/text with empty strings
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.strip()

# Remove articles with no text and duplicate articles
df = df[df["text"] != ""].copy()
df = df.drop_duplicates(subset=["text"]).copy()

print("Shape after cleaning missing text and duplicates:", df.shape)

Shape after cleaning missing text and duplicates: (62704, 4)


### Basic test cleaning

In [44]:
df["full_text"] = (
    df["title"] + " " + df["text"]
).str.strip()

def basic_clean(text):
    text = text.lower()
    # Remove HTML
    text = re.sub(r"<[^>]+>", " ", text)
    # Replace URLs with a token
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    # Remove numbers
    text = re.sub(r"\d+", " ", text)
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["full_text"].apply(basic_clean)

### Tokenization and Lemmatization

In [45]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = text.split()
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

df["tokens"] = df["clean_text"].apply(preprocess_text)

# Convert tokens back into text for ML models
df["processed_text"] = df["tokens"].apply(lambda x: " ".join(x))

### Additional Features

In [46]:
df["word_count"] = df["full_text"].str.split().str.len()
df["char_count"] = df["full_text"].str.len()
df["sentence_count"] = df["full_text"].str.count(r"[.!?]")
df["url_count"] = df["full_text"].str.count(
    r"https?://\S+|www\.\S+"
)
df["exclamation_count"] = df["full_text"].str.count("!")
df["question_count"] = df["full_text"].str.count(r"\?")

In [47]:
display(
    df[
        [
            "title",
            "text",
            "processed_text",
            "label",
            "word_count",
            "char_count",
            "sentence_count",
            "url_count",
            "exclamation_count",
            "question_count"
        ]
    ].head()
)

,title,text,processed_text,label,word_count,char_count,sentence_count,url_count,exclamation_count,question_count
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,law enforcement high alert following threat co...,1,889,5180,63,0,2,7
1,,Did they post their votes for Hillary already?,post vote hillary already,1,8,46,1,0,0,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last n...",unbelievable obama’s attorney general say char...,1,52,353,2,0,1,0
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,bobby jindal raised hindu us story christian c...,0,1337,8116,61,0,0,1
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",satan russia unvelis image terrifying new ‘sup...,1,345,2012,11,0,0,0


In [48]:
# Saving the data so that we can also compare our results with the raw and the preprocessed data!

Path("../data/processed").mkdir(parents=True, exist_ok=True)

df.to_csv(
    "../data/processed/WELFake_processed.csv",
    index=False
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.
